<a href="https://colab.research.google.com/github/giannismantzaris-cmd/DAMA61/blob/main/Mantzaris_WA5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# increase the width of the notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [20]:
#All needed imports
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

# for reproducibility
tf.random.set_seed(42)

In [7]:
from tensorflow.keras.datasets import cifar10
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

In [8]:
print("Original training set shape:", x_train_full.shape, y_train_full.shape)
print("Original test set shape:    ", x_test.shape, y_test.shape)

Original training set shape: (50000, 32, 32, 3) (50000, 1)
Original test set shape:     (10000, 32, 32, 3) (10000, 1)


In [9]:
# Normalize the pixel values to [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [10]:
# Combine train and test to split as 60/20/20
X = np.concatenate([x_train_full, x_test], axis=0)
Y = np.concatenate([y_train_full, y_test], axis=0)
print("Full dataset:", X.shape, Y.shape)

Full dataset: (60000, 32, 32, 3) (60000, 1)


In [11]:
# Shuffle
np.random.seed(42)
indices = np.random.permutation(len(X))
X = X[indices]
Y = Y[indices]

In [14]:
# Split the full dateset to: 60% / 20% / 20%
n_total = len(X)  # 60000
n_train = int(0.6 * n_total)  # 36000
n_valid_and_test = int(0.2 * n_total)  # 12000

X_train = X[:n_train]
y_train = Y[:n_train]

X_valid = X[n_train:n_train + n_valid_and_test]
y_valid = Y[n_train:n_train + n_valid_and_test]

X_test = X[n_train + n_valid_and_test:]
y_test = Y[n_train + n_valid_and_test:]

In [15]:
# Print shapes to verify
print("Training set:   ", X_train.shape, y_train.shape)
print("Validation set: ", X_valid.shape, y_valid.shape)
print("Test set:       ", X_test.shape, y_test.shape)

Training set:    (36000, 32, 32, 3) (36000, 1)
Validation set:  (12000, 32, 32, 3) (12000, 1)
Test set:        (12000, 32, 32, 3) (12000, 1)


In [18]:
from tensorflow.keras.utils import to_categorical
num_classes = 10

# Convert to one-hot
y_train_oh = to_categorical(y_train, num_classes)
y_valid_oh   = to_categorical(y_valid, num_classes)
y_test_oh  = to_categorical(y_test, num_classes)

In [19]:
print("y_train one-hot:", y_train_oh.shape)
print("y_val one-hot:", y_valid_oh.shape)
print("y_test one-hot:", y_test_oh.shape)

y_train one-hot: (36000, 10)
y_val one-hot: (12000, 10)
y_test one-hot: (12000, 10)


In [26]:
model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
#first layer
    layers.Conv2D(filters=16, kernel_size=(5, 5), strides=2, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
#second layer - retain omage size by adding padding zeros
    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
#third layer - retain imagr size by adding padding zeros
    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

#classification part
    layers.Flatten(),
    layers.Dropout(0.25),

    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(10, activation="softmax")
])